# 第 14 章: K-means によるクラスタリングの探索と可視化

エルボー法でクラスタ数を選び、Tribuo の SSE と比べて、クラスタごとの特徴を確認する。

In [ ]:
%use dataframe(0.15.0), kandy(0.8.0)
@file:DependsOn("org.tribuo:tribuo-clustering-kmeans:4.3.2")

In [ ]:
@file:DependsOn("../build/libs/getting-started-ml.jar")

In [ ]:
import chapter14.kmeansWithRestarts
import chapter14.loadSpending
import chapter14.sseByClusterCount
import chapter14.standardize
import chapter14.summarizeClusters
import chapter14.tribuoBestSse
import java.io.File
import java.util.logging.Level
import java.util.logging.Logger

// Tribuo が学習の経過を標準エラーに出すので、警告以上だけにする
Logger.getLogger("org.tribuo").level = Level.WARNING

// Notebook は notebooks/ で実行されるので、学習データの既定の場所を 1 つ上にずらす
val wholesaleCsv = File(dataset.dataDir { name -> System.getenv(name) ?: "../../data/sukkiri-ml" }, "Wholesale.csv")
val df = loadSpending(wholesaleCsv)
val points = standardize(df)
df.rowsCount() to df.columnsCount()

## エルボー法

In [ ]:
val counts = (1..10).toList()
val mine = sseByClusterCount(points, counts, seed = 0)
val tribuo = counts.associateWith { tribuoBestSse(points, it, seed = 0L) }
val elbow =
    dataFrameOf(
        "クラスタ数" to counts + counts,
        "実装" to counts.map { "自作" } + counts.map { "Tribuo（k-means++）" },
        "SSE" to counts.map { mine.getValue(it) } + counts.map { tribuo.getValue(it) },
    )
elbow.plot {
    line {
        x("クラスタ数")
        y("SSE")
        color("実装")
    }
    points {
        x("クラスタ数")
        y("SSE")
        color("実装")
    }
    layout.title = "エルボー法"
}

In [ ]:
dataFrameOf(
    "クラスタ数" to counts,
    "自作" to counts.map { mine.getValue(it) },
    "Tribuo（k-means++）" to counts.map { tribuo.getValue(it) },
    "自作の減少量" to counts.map { k -> if (k == 1) null else mine.getValue(k - 1) - mine.getValue(k) },
)

## クラスタごとの特徴

In [ ]:
val result = kmeansWithRestarts(points, nClusters = 5, seed = 0)
val summary = summarizeClusters(df, result.labels)
dataFrameOf(
    listOf(
        summary.map { it.cluster }.toColumn("クラスタ"),
        summary.map { it.count }.toColumn("件数"),
    ) + df.columnNames().map { column -> summary.map { Math.round(it.means.getValue(column)) }.toColumn(column) },
)

In [ ]:
val centers =
    dataFrameOf(
        "クラスタ" to result.centers.indices.flatMap { k -> df.columnNames().map { k.toString() } },
        "列" to result.centers.indices.flatMap { df.columnNames() },
        "中心（標準化後）" to result.centers.flatMap { it },
    )
centers.plot {
    bars {
        x("列")
        y("中心（標準化後）")
        fillColor("クラスタ")
    }
    layout.title = "クラスタごとの中心（標準化後）"
}

In [ ]:
val clustered = df.add("クラスタ") { result.labels[index()].toString() }
clustered.plot {
    points {
        x("Grocery")
        y("Fresh")
        color("クラスタ")
    }
    layout.title = "Grocery と Fresh の支出額（クラスタ別）"
}